---
# <div align="center"><font color='red'>  </font></div>
# <div align="center"><font color='red'> COSC 2779/2972 | Deep Learning  </font></div>
## <div align="center"> <font color='red'> Week 3 Lectorial Example: **Neural Network Optimisation**</font></div>
---

# Linear Regression Example

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

import torch
import torch.nn as nn

from IPython.display import display, HTML
import matplotlib.pyplot as plt
import matplotlib.animation as animation
%matplotlib inline


## Setting up notebook and Loading Data

Place `height_v_weight.csv` in the same directory as this notebook (no Google Drive mount needed).

In [ ]:
df = pd.read_csv('height_v_weight.csv')
df = df.drop(labels=' Gender', axis=1)
df.head(5)


Lets plot the data

In [ ]:
def draw_scatter(x, y, slope=None, intercept=None, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))

    sns.scatterplot(x=x, y=y, ax=ax, label='Data')
    ax.set_xlabel('Height (inches)')
    ax.set_ylabel('Weight (lbs)')

    h0, h1 = np.min(x) - 0.1 * (np.max(x) - np.min(x)), np.max(x) + 0.1 * (np.max(x) - np.min(x))
    w0, w1 = np.min(y) - 0.1 * (np.max(y) - np.min(y)), np.max(y) + 0.1 * (np.max(y) - np.min(y))
    ax.set_xlim(h0, h1)
    ax.set_ylim(w0, w1)
    ax.set_aspect((h1 - h0) / (w1 - w0))

    if slope is not None and intercept is not None:
        pred_w0, pred_w1 = (slope * h0 + intercept, slope * h1 + intercept)
        ax.plot([h0, h1], [pred_w0, pred_w1], 'r-.', label='Fitted Line')

    ax.legend(loc='upper left')
    return ax


In [ ]:
_ = draw_scatter(df['Height'].values, df['Weight'].values)


## Setting up the problem

We need a model that can predict the weight of a person given Height. By observing the data we can assume that a linear model will be adequate for the task. Therefore, lets pick a linear hypothesis space defined by:

$$ y_i \approx h(x_i; w,b) = w \times x_i + b$$

Now we need to establish a performance measure to determine if a given hypothesis is ''good''. We can use mean squared error:

$$MSE = (y_i - h(x_i; w,b))^2$$

**How to find the best hypothesis ($h^*$) in this hypothesis space?**

In [ ]:
# Lets create a meshgrid for w and b
w_space = np.linspace(-10, 10, 100)
b_space = np.linspace(-200, 200, 100)
w_mesh, b_mesh = np.meshgrid(w_space, b_space)

# Lets calculate the loss for each point in the meshgrid
def loss_function(w, b, x, y):
    y_hat = w * x + b
    loss = np.mean((y - y_hat) ** 2)
    return loss

loss_mesh = np.zeros_like(w_mesh)
for i in range(w_mesh.shape[0]):
    for j in range(w_mesh.shape[1]):
        loss_mesh[i, j] = loss_function(w_mesh[i, j], b_mesh[i, j], df['Height'].values, df['Weight'].values)

# Lets plot the loss surface
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(w_mesh, b_mesh, loss_mesh, cmap='viridis')
ax.set_xlabel('w')
ax.set_ylabel('b')
ax.set_zlabel('Loss')
ax.set_title('Loss Surface')
plt.show()


In [ ]:
from sklearn.preprocessing import MinMaxScaler

slope = np.arange(-1.0, 2.0, 0.1)
bias = np.arange(-1.0, 1.0, 0.001)
w0, w1 = np.meshgrid(slope, bias)

X = df['Height'].values
y = df['Weight'].values

scaler = MinMaxScaler()
X = scaler.fit_transform(X.reshape(-1, 1)).reshape(-1,)
y = scaler.fit_transform(y.reshape(-1, 1)).reshape(-1,)

loss = np.zeros_like(w0)
for i in range(0, w0.shape[0]):
    for j in range(0, w0.shape[1]):
        ypred = w0[i, j] * X + w1[i, j]
        loss[i, j] = np.mean(np.power((y - ypred), 2))

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
surf = ax.plot_surface(w0, w1, loss, label="Loss surface", cmap='viridis', edgecolor='none')
surf._facecolors2d = surf._facecolor3d
surf._edgecolors2d = surf._edgecolor3d
ax.set_xlabel('Slope')
ax.set_ylabel('Bias')
ax.legend()
plt.show()


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 5))
draw_scatter(X, y, slope=1, intercept=0.1, ax=axs[0])
draw_scatter(X, y, slope=2, intercept=0.0, ax=axs[1])
draw_scatter(X, y, slope=-1, intercept=1, ax=axs[2])
plt.tight_layout()


How to search for the best model parameters that minimise the loss?

- Random search - Bad idea.
- Solve $\nabla_{\mathbf{w}} \mathcal{L\left ( \mathbf{w} \right )}   = 0$ and find a closed form solution. Works for the simple problem we have. Not applicable for many cases.
- Guided Search - Apply iterative method - Gradient descent:

$$\mathbf{w}^{[t]} = \mathbf{w}^{[t-1]} - \alpha~ \nabla_{\mathbf{w}} \mathcal{L}\left ( \mathbf{w} \right )$$

## Analytical solution

Lets fit a linear model with mean squared error as the loss. For this problem we have a closed form solution (analytical solution).

Let's find the optimal model parameters using analytical solution with sklearn.

We will assume this solution as the true underlying function.

In [ ]:
def calc_mean_sq_error(x, y, slope, intercept):
    diffs = (slope * x + intercept) - y
    mse = np.mean(diffs ** 2)
    return mse


In [ ]:
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()
lin_reg.fit(X.reshape([-1, 1]), y)

true_slope, true_intercept = lin_reg.coef_[0], lin_reg.intercept_
min_mse = calc_mean_sq_error(X, y, true_slope, true_intercept)

_ = draw_scatter(X, y, slope=true_slope, intercept=true_intercept)


In [ ]:
print(true_slope, true_intercept, min_mse)


## Run Gradient descent

We now replace TensorFlow's `GradientTape` + `keras.optimizers` with PyTorch's autograd + `torch.optim`. The model is still a simple linear regression: `slope` and `intercept` are `torch.nn.Parameter`s we optimise directly.

In [ ]:
n_iter = 150

def run_gradient_descent(X_, y_, opti='sgd', n_iter=n_iter, lr=0.01):
    loss_hist = []
    slope_hist = []
    intercep_hist = []

    X_t = torch.tensor(X_, dtype=torch.float32)
    y_t = torch.tensor(y_, dtype=torch.float32)

    slope_ = nn.Parameter(torch.empty(1).uniform_(-1, 1))
    intercep_ = nn.Parameter(torch.empty(1).uniform_(0, 1))
    trainable_params = [slope_, intercep_]

    if opti == 'sgd':
        optimizer = torch.optim.SGD(trainable_params, lr=lr)
    elif opti == 'RMSProp':
        optimizer = torch.optim.RMSprop(trainable_params, lr=lr)

    for i in range(n_iter):
        optimizer.zero_grad()

        predictions = slope_ * X_t + intercep_
        errors = predictions - y_t
        loss = torch.mean(errors ** 2)

        loss.backward()
        optimizer.step()

        loss_hist.append(loss.item())
        slope_hist.append(slope_.item())
        intercep_hist.append(intercep_.item())

    return loss_hist, slope_hist, intercep_hist


In [ ]:
loss_hist, slope_hist, intercep_hist = run_gradient_descent(X, y)


plot results

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 4))
draw_scatter(X, y, slope=None, intercept=None, ax=axs[0])

hmin, hmax = 0, len(loss_hist)
wmin, wmax = 0, np.max(loss_hist)
axs[1].plot([hmin, hmax], [min_mse, min_mse], 'g-.', label='Minimum Possible Loss')
axs[1].set_xlim(hmin, hmax)
axs[1].set_ylim(wmin, wmax)
axs[1].set_xlabel('Iteration')
axs[1].set_ylabel('Loss (MSE)')
axs[1].legend(loc='upper right')
axs[1].set_aspect(hmax / float(wmax))

txt_title = axs[0].set_title('')

min_loss, = axs[1].plot([], [], 'g-.', label='Minimum Possible Loss')
loss_line, = axs[1].plot([], [], 'b-', marker='o', label='Computed Loss')
fit_line, = axs[0].plot([], [], 'r-.')

h0, h1 = np.min(X) - 0.1 * (np.max(X) - np.min(X)), np.max(X) + 0.1 * (np.max(X) - np.min(X))

def animate_plot(i):
    pred_w0, pred_w1 = (slope_hist[i] * h0 + intercep_hist[i], slope_hist[i] * h1 + intercep_hist[i])
    fit_line.set_data([h0, h1], [pred_w0, pred_w1])
    loss_line.set_data(range(len(loss_hist[:i])), loss_hist[:i])
    min_loss.set_data([hmin, hmax], [min_mse, min_mse])
    txt_title.set_text('Frame = {0:2f}, {0:2f}'.format(intercep_hist[i], slope_hist[i]))
    return fit_line, loss_line, min_loss


In [ ]:
anim = animation.FuncAnimation(fig, animate_plot, frames=n_iter - 1, interval=200, blit=True)


In [ ]:
HTML(anim.to_html5_video())


## Run Stochastic Gradient descent

Calculating the gradients of the empirical risk.

$$ \nabla_{\mathbf{w}} \mathcal{L}\left ( \mathbf{w} \right ) = \nabla_{\mathbf{w}} {\mathbb{E}_{\left(\mathbf{x},y\right) \sim \hat{p}_{data}} \textrm{L} \left( y, h \left( \mathbf{x}; \mathbf{w}\right)\right)} = \frac{1}{N} \sum_{i=1}^N \nabla_{\mathbf{w}} \textrm{L} \left( y^{(i)}, h \left( \mathbf{x}^{(i)}; \mathbf{w}\right)\right)
$$

Computing this expectation exactly is very expensive because it requires evaluating the model on every example in the dataset. In practice we estimate this expectation using a small, randomly sampled batch of examples at each step — Stochastic Gradient Descent.

In [ ]:
def run_stochastic_gradient_descent(X_, y_, opti='sgd', n_iter=n_iter, batch_size=32, lr=0.01):
    loss_hist = []
    slope_hist = []
    intercep_hist = []

    slope_ = nn.Parameter(torch.empty(1).uniform_(-1, 1))
    intercep_ = nn.Parameter(torch.empty(1).uniform_(0, 1))
    trainable_params = [slope_, intercep_]

    if opti == 'sgd':
        optimizer = torch.optim.SGD(trainable_params, lr=lr)
    elif opti == 'RMSProp':
        optimizer = torch.optim.RMSprop(trainable_params, lr=lr)

    for i in range(n_iter):
        idx_batch = np.random.choice(len(X_), batch_size)
        X_batch = torch.tensor(X_[idx_batch], dtype=torch.float32)
        y_batch = torch.tensor(y_[idx_batch], dtype=torch.float32)

        optimizer.zero_grad()

        predictions = slope_ * X_batch + intercep_
        errors = predictions - y_batch
        loss = torch.mean(errors ** 2)

        loss.backward()
        optimizer.step()

        loss_hist.append(loss.item())
        slope_hist.append(slope_.item())
        intercep_hist.append(intercep_.item())

    return loss_hist, slope_hist, intercep_hist


In [ ]:
loss_hist, slope_hist, intercep_hist = run_stochastic_gradient_descent(X, y)


Plot results

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 4))
draw_scatter(X, y, slope=None, intercept=None, ax=axs[0])

hmin, hmax = 0, len(loss_hist)
wmin, wmax = 0, np.max(loss_hist)
axs[1].plot([hmin, hmax], [min_mse, min_mse], 'g-.', label='Minimum Possible Loss')
axs[1].set_xlim(hmin, hmax)
axs[1].set_ylim(wmin, wmax)
axs[1].set_xlabel('Iteration')
axs[1].set_ylabel('Loss (MSE)')
axs[1].legend(loc='upper right')
axs[1].set_aspect(hmax / float(wmax))

txt_title = axs[0].set_title('')

min_loss, = axs[1].plot([], [], 'g-.', label='Minimum Possible Loss')
loss_line, = axs[1].plot([], [], 'b-', marker='o', label='Computed Loss')
fit_line, = axs[0].plot([], [], 'r-.')

h0, h1 = np.min(X) - 0.1 * (np.max(X) - np.min(X)), np.max(X) + 0.1 * (np.max(X) - np.min(X))

def animate_plot(i):
    pred_w0, pred_w1 = (slope_hist[i] * h0 + intercep_hist[i], slope_hist[i] * h1 + intercep_hist[i])
    fit_line.set_data([h0, h1], [pred_w0, pred_w1])
    loss_line.set_data(range(len(loss_hist[:i])), loss_hist[:i])
    min_loss.set_data([hmin, hmax], [min_mse, min_mse])
    txt_title.set_text('Frame = {0:2f}, {0:2f}'.format(intercep_hist[i], slope_hist[i]))
    return fit_line, loss_line, min_loss


In [ ]:
anim = animation.FuncAnimation(fig, animate_plot, frames=n_iter - 1, interval=200, blit=True)


In [ ]:
HTML(anim.to_html5_video())


## Improved optimiser

### Deterministic RMSProp

In [ ]:
loss_hist, slope_hist, intercep_hist = run_gradient_descent(X, y, opti='RMSProp')


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 4))
draw_scatter(X, y, slope=None, intercept=None, ax=axs[0])

hmin, hmax = 0, len(loss_hist)
wmin, wmax = 0, np.max(loss_hist)
axs[1].plot([hmin, hmax], [min_mse, min_mse], 'g-.', label='Minimum Possible Loss')
axs[1].set_xlim(hmin, hmax)
axs[1].set_ylim(wmin, wmax)
axs[1].set_xlabel('Iteration')
axs[1].set_ylabel('Loss (MSE)')
axs[1].legend(loc='upper right')
axs[1].set_aspect(hmax / float(wmax))

txt_title = axs[0].set_title('')

min_loss, = axs[1].plot([], [], 'g-.', label='Minimum Possible Loss')
loss_line, = axs[1].plot([], [], 'b-', marker='o', label='Computed Loss')
fit_line, = axs[0].plot([], [], 'r-.')

h0, h1 = np.min(X) - 0.1 * (np.max(X) - np.min(X)), np.max(X) + 0.1 * (np.max(X) - np.min(X))

def animate_plot(i):
    pred_w0, pred_w1 = (slope_hist[i] * h0 + intercep_hist[i], slope_hist[i] * h1 + intercep_hist[i])
    fit_line.set_data([h0, h1], [pred_w0, pred_w1])
    loss_line.set_data(range(len(loss_hist[:i])), loss_hist[:i])
    min_loss.set_data([hmin, hmax], [min_mse, min_mse])
    txt_title.set_text('Frame = {0:2f}, {0:2f}'.format(intercep_hist[i], slope_hist[i]))
    return fit_line, loss_line, min_loss


In [ ]:
anim = animation.FuncAnimation(fig, animate_plot, frames=n_iter - 1, interval=200, blit=True)


In [ ]:
HTML(anim.to_html5_video())


### Stochastic RMSProp

In [ ]:
loss_hist, slope_hist, intercep_hist = run_stochastic_gradient_descent(X, y, opti='RMSProp')


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 4))
draw_scatter(X, y, slope=None, intercept=None, ax=axs[0])

hmin, hmax = 0, len(loss_hist)
wmin, wmax = 0, np.max(loss_hist)
axs[1].plot([hmin, hmax], [min_mse, min_mse], 'g-.', label='Minimum Possible Loss')
axs[1].set_xlim(hmin, hmax)
axs[1].set_ylim(wmin, wmax)
axs[1].set_xlabel('Iteration')
axs[1].set_ylabel('Loss (MSE)')
axs[1].legend(loc='upper right')
axs[1].set_aspect(hmax / float(wmax))

txt_title = axs[0].set_title('')

min_loss, = axs[1].plot([], [], 'g-.', label='Minimum Possible Loss')
loss_line, = axs[1].plot([], [], 'b-', marker='o', label='Computed Loss')
fit_line, = axs[0].plot([], [], 'r-.')

h0, h1 = np.min(X) - 0.1 * (np.max(X) - np.min(X)), np.max(X) + 0.1 * (np.max(X) - np.min(X))

def animate_plot(i):
    pred_w0, pred_w1 = (slope_hist[i] * h0 + intercep_hist[i], slope_hist[i] * h1 + intercep_hist[i])
    fit_line.set_data([h0, h1], [pred_w0, pred_w1])
    loss_line.set_data(range(len(loss_hist[:i])), loss_hist[:i])
    min_loss.set_data([hmin, hmax], [min_mse, min_mse])
    txt_title.set_text('Frame = {0:2f}, {0:2f}'.format(intercep_hist[i], slope_hist[i]))
    return fit_line, loss_line, min_loss


In [ ]:
anim = animation.FuncAnimation(fig, animate_plot, frames=n_iter - 1, interval=200, blit=True)


In [ ]:
HTML(anim.to_html5_video())


# Simple network

Now let's move from a single linear unit to a small feed-forward network, trained on the Iris dataset (4 features, 3 classes).

In [ ]:
from sklearn import datasets, preprocessing, model_selection

torch.manual_seed(42)

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 10)
        self.fc2 = nn.Linear(10, 10)
        self.fc3 = nn.Linear(10, 3)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)  # logits - nn.CrossEntropyLoss applies softmax internally

# 2. get data
data = datasets.load_iris()
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    data["data"], data["target"], test_size=0.25, random_state=1)

scaler_x = preprocessing.MinMaxScaler(feature_range=(-1, 1)).fit(X_train)
X_train = scaler_x.transform(X_train)
X_test = scaler_x.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

print("Shape of X_train:", X_train_t.shape)
print("Shape of y_train:", y_train_t.shape)


In [ ]:
def train_model(weight_decay=0.0, epochs=60, batch_size=32, lr=0.1, seed=42):
    torch.manual_seed(seed)
    model = SimpleNet()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    train_loss_hist, test_loss_hist = [], []
    n = len(X_train_t)
    for epoch in range(epochs):
        perm = torch.randperm(n)
        epoch_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            xb, yb = X_train_t[idx], y_train_t[idx]

            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * len(xb)

        train_loss_hist.append(epoch_loss / n)
        with torch.no_grad():
            test_loss_hist.append(criterion(model(X_test_t), y_test_t).item())

    return model, train_loss_hist, test_loss_hist


# 3. train model
model, train_loss_hist, test_loss_hist = train_model()

# 4. evaluate + plot the training curve (replaces the TF loss-landscape visualisation)
with torch.no_grad():
    test_acc = (model(X_test_t).argmax(1) == y_test_t).float().mean().item()
print(f"Test accuracy: {test_acc:.3f}")

plt.figure(figsize=(6, 4))
plt.plot(train_loss_hist, label='Train loss')
plt.plot(test_loss_hist, label='Test loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('Simple network training curve')
plt.legend()
plt.show()


# Regularisation

Same network, but with L2 weight decay applied via the optimiser's `weight_decay` argument. Compare the training curves and test accuracy with and without regularisation.

In [ ]:
model_l2, train_loss_hist_l2, test_loss_hist_l2 = train_model(weight_decay=0.01)

with torch.no_grad():
    test_acc_l2 = (model_l2(X_test_t).argmax(1) == y_test_t).float().mean().item()

print(f"No regularisation - test accuracy: {test_acc:.3f}")
print(f"L2 regularisation  - test accuracy: {test_acc_l2:.3f}")

plt.figure(figsize=(6, 4))
plt.plot(test_loss_hist, label='No regularisation - test loss')
plt.plot(test_loss_hist_l2, label='L2 regularisation - test loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('Effect of L2 regularisation on test loss')
plt.legend()
plt.show()
